In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## **Data Understanding & Schema**

In [4]:
# 1. Load the raw dataset
df = pd.read_csv("/logistics_pricing_dataset.csv")
print(df.head())


  Shipment_ID       Origin  Destination  Distance_Miles  Weight_Lbs  \
0   SHP100000      Houston        Miami          1691.0     10752.0   
1   SHP100001      Atlanta      Houston          1798.0      8478.0   
2   SHP100002  Los Angeles      Houston          2218.0      9386.0   
3   SHP100003      Atlanta      Chicago          1731.0     14891.0   
4   SHP100004      Atlanta  Los Angeles          2371.0      3303.0   

  Transport_Mode Carrier  Lead_Time_Days  Fuel_Surcharge_Rate  Is_Urgent  \
0      Truckload     UPS               9                 0.19          0   
1      Truckload    USPS              13                 0.36          0   
2            LTL   FedEx               6                 0.27          0   
3            LTL    USPS               6                 0.30          0   
4      Truckload   FedEx               4                 0.42          1   

   Actual_Price_USD  
0           4830.69  
1           4811.85  
2           4142.41  
3           3884.64  
4     

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Shipment_ID          1000 non-null   object 
 1   Origin               1000 non-null   object 
 2   Destination          1000 non-null   object 
 3   Distance_Miles       1000 non-null   float64
 4   Weight_Lbs           1000 non-null   float64
 5   Transport_Mode       1000 non-null   object 
 6   Carrier              1000 non-null   object 
 7   Lead_Time_Days       1000 non-null   int64  
 8   Fuel_Surcharge_Rate  1000 non-null   float64
 9   Is_Urgent            1000 non-null   int64  
 10  Actual_Price_USD     1000 non-null   float64
 11  Predicted_Price_USD  1000 non-null   float64
dtypes: float64(5), int64(2), object(5)
memory usage: 93.9+ KB




The generated dataset contains the following variables that dictate real-world shipping rates:

- **Shipment_ID:** Unique string identifier for tracking.

- **Origin / Destination:** Regional distribution hubs (Chicago, New York, Los Angeles, Houston, Atlanta, Miami).

- **Distance_Miles** haul route distance (contains dirty -999 placeholder outliers).

- **Weight_Lbs:** Total cargo weight (contains NaN missing values).

- **Transport_Mode:** Fleet segmentation (LTL, Truckload, Air Freight).

- **Carrier:** Logistical service partners handling the shipment

- **Lead_Time_Days:** Booking notice delay window.

- **Fuel_Surcharge_Rate:** Dynamic matrix floating cost fee factor.

- **Is_Urgent:** Express shipping status flag (1 = high-priority, 0 = standard timeline).

- **Actual_Price_USD:** Final invoice cost (Machine Learning Target Variables).

## **Data Cleaning and Preprocessing Pipeline**

In [5]:
# 2. Handle dirty placeholders: Convert spatial distance outliers (-999) to NaN
df['Distance_Miles'] = df['Distance_Miles'].replace(-999, np.nan)

# 3. Separate features and target price matrix
X = df.drop(columns=['Shipment_ID', 'Actual_Price_USD'])
y = df['Actual_Price_USD']

# 4. Define specific feature columns
numerical_features = ['Distance_Miles', 'Weight_Lbs', 'Lead_Time_Days', 'Fuel_Surcharge_Rate', 'Is_Urgent']
categorical_features = ['Origin', 'Destination', 'Transport_Mode', 'Carrier']

# 5. Build isolated pipeline blocks
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))  # Safely fills missing Weight & Distance with median
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Encodes cities and carrier flags
])

# 6. Combine feature transformations into a structural preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# ***Building and Training the ML Model***

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [7]:
# 1. Attach model estimator to our structural transformation pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

In [8]:
# 2. Create Train-Test split validation matrices (80% Train, 20% Evaluation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 3. Train the machine learning pricing engine
model_pipeline.fit(X_train, y_train)

# 4. Evaluate predictive performance metrics
y_pred = model_pipeline.predict(X_test)
print(f"Model R² Score (Variance Explained): {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error (Average Cost Deviation): ${mean_absolute_error(y_test, y_pred):.2f}")

# 5. Export structural prediction columns to populate dashboard visualization views
df['Predicted_Price_USD'] = model_pipeline.predict(X)
# Backfill raw file missing items to prevent blank dashboard displays
df['Distance_Miles'] = df['Distance_Miles'].fillna(df['Distance_Miles'].median())
df['Weight_Lbs'] = df['Weight_Lbs'].fillna(df['Weight_Lbs'].median())

df.to_csv("dashboard_pricing_data.csv", index=False)
print("Data exported successfully!")

Model R² Score (Variance Explained): 0.9848
Mean Absolute Error (Average Cost Deviation): $164.20
Data exported successfully!


In [21]:
print(df.head())


  Shipment_ID       Origin  Destination  Distance_Miles  Weight_Lbs  \
0   SHP100000      Houston        Miami          1691.0     10752.0   
1   SHP100001      Atlanta      Houston          1798.0      8478.0   
2   SHP100002  Los Angeles      Houston          2218.0      9386.0   
3   SHP100003      Atlanta      Chicago          1731.0     14891.0   
4   SHP100004      Atlanta  Los Angeles          2371.0      3303.0   

  Transport_Mode Carrier  Lead_Time_Days  Fuel_Surcharge_Rate  Is_Urgent  \
0      Truckload     UPS               9                 0.19          0   
1      Truckload    USPS              13                 0.36          0   
2            LTL   FedEx               6                 0.27          0   
3            LTL    USPS               6                 0.30          0   
4      Truckload   FedEx               4                 0.42          1   

   Actual_Price_USD  Predicted_Price_USD  
0           4830.69            4845.6731  
1           4811.85           

#***Insights Through Interactive Dashboard***

In [12]:
# Install Streamlit and localtunnel execution utilities
!pip install -q streamlit plotly
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [25]:
%%writefile /content/pricing.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(layout="wide")
st.title("🚚 Logistics Pricing Optimization Engine")

# Load compiled predictive file
df = pd.read_csv("dashboard_pricing_data.csv")

# --- SIDEBAR INTERACTIVE FILTERS & SLICERS ---
st.sidebar.header("Dashboard Slicers")
selected_origin = st.sidebar.multiselect("Origin Hub", options=df['Origin'].unique(), default=df['Origin'].unique())
selected_mode = st.sidebar.multiselect("Transport Mode", options=df['Transport_Mode'].unique(), default=df['Transport_Mode'].unique())
weight_slider = st.sidebar.slider("Maximum Cargo Weight (Lbs)", int(df['Weight_Lbs'].min()), int(df['Weight_Lbs'].max()), int(df['Weight_Lbs'].max()))

# Filter dataset dynamically based on UI selection
filtered_df = df[
    (df['Origin'].isin(selected_origin)) &
    (df['Transport_Mode'].isin(selected_mode)) &
    (df['Weight_Lbs'] <= weight_slider)
]

# --- DASHBOARD METRICS ---
col1, col2, col3 = st.columns(3)
col1.metric("Total Shipments Routed", len(filtered_df))
col2.metric("Avg Actual Price Paid", f"${filtered_df['Actual_Price_USD'].mean():.2f}")
col3.metric("Avg Predicted Price Model", f"${filtered_df['Predicted_Price_USD'].mean():.2f}")

# --- DASHBOARD GRAPH VISUALIZATIONS ---
st.subheader("Model Performance: Actual Costs vs. Model Projections")
fig = px.scatter(filtered_df, x="Actual_Price_USD", y="Predicted_Price_USD", color="Transport_Mode",
                 hover_data=["Origin", "Destination", "Distance_Miles"],
                 labels={"Actual_Price_USD": "Actual Invoice ($)", "Predicted_Price_USD": "Model Forecast ($)"})
st.plotly_chart(fig, use_container_width=True)


Overwriting /content/pricing.py
